In [ ]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [ ]:
%run /Workspace/consolidated_pipeline/1_setup/utilities

In [ ]:
dbutils.widgets.text('catalog', 'fmcg', 'Catalog')
dbutils.widgets.text('data_source', 'products', 'Data Source')

catalog = dbutils.widgets.get('catalog')
data_source = dbutils.widgets.get('data_source')

base_path = f's3://sports-consolidated-dp./{data_source}/*.csv'


In [ ]:
# ---> BRONZE <---

df = (
    spark.read.format('csv')
    .option('header', True)
    .option('inferSchema', True)
    .load(base_path)
    .withColumn('read_timestamp', F.current_timestamp())
    .select('*', '_metadata.file_name', '_metadata.file_size')
)



In [ ]:
df.printSchema()
display(df.limit(10))

In [ ]:
df.write\
    .format('delta') \
    .option('delta.enableChangeDataFeed', 'true') \
    ,mode('overwrite') \
    .saveAsTable(f'{catalog}.{bronze_schema}.{data_souce}')

In [ ]:
# ---> SILVER <---
df_bronze = spark.sql(f'SELECT * FROM {catalog}.{bronze_schema}.{data_source};')
df_bronze.show(10)

In [ ]:
# ---> DROP DUPLICATES <---
print('Rows before duplicates dropped: ', df_bronze.count())
df_silver = df_bronze.dropDuplicates(['product_id'])
print('Rows after duplicates dropped: ', df_silver.count())

In [ ]:
# ---> TITLE FIX CASE <---
# (energy bars ---> Energy Bars, protien bars ---> Protien Bars etc)

df_silver.select(category).distinct().show()

In [ ]:
df_silver = df_silver.withColumn(
    'category',
    F.when(F.col('category').isNull(), None)
    .otherwise(F.initcap('category'))
)

df_silver.select('category').distinct().show()

In [ ]:
# Replace 'protien' -> 'Protein' in both product_name and category
df_silver = (
    df_silver
    .withColumn(
        'product_name',
        F.regexp_replace(F.col('product_name'), '(?i)Protien', 'Protein')
    )
    .withColumn(
        'category',
        F.regexp_replace(F.col('category'), '(?)iProtien', 'Protein')
    )
)

display(df_silver.limit(10))

In [ ]:
# --> ADD DIVISON COLUMN <--
df_silver = (
    df_silver
    .withColumn(
        'division',
        F.when(F.col('category') == 'Energy Bars',          'Nutrition Bars')
         .when(F.col('category') == 'Protein Bars',         'Nutrition Bars')
         .when(F.col('category') == 'Granola & Cereals',    'Breakfast Foods')
         .when(F.col('category') == 'Recovery Dairy',       'Dairy & Recovery')
         .when(F.col('category') == 'Healthy Snacks',       'Healthy Snacks')
         .when(F.col('category') == 'Electrolyte Mix',      'Hydration & Electrolytes')
         .otherwise('Other')
    )
)

# --> ADD VARIANT COLUMN <--
# Extract weight from product_name SportsBar Protein Bar Peanut Crunch (45g)
df_silver = df_silver.withColumn(
    'variant',
    F.regexp_extract(F.col('product_name', r"\((.*?)\)", 1))
)

# Invalid product_ids are replaced with a fallback value to avoid losing fact records and ensure downstream joins remain consistent

df_silver = (
    df_silver
    .withColumn(
        'product_code',
        F.sha2(F.col('product_id').cast('string'), 256)
    )
    .withColumn(
        'product_id',
        F.when(
            F.col('product_id').cast('string').rlike('^[0-9]+$'),
            F.col('product_id').cast('string')
        )
        .otherwise(F.lit(999999).cast('string'))
    )
    # Rename product_name -> product
    .withColumnRenamed('product_name', 'product')
)

display(df_silver)

In [ ]:
df_silver = df_silver.select('product_code', 'division', 'category', 'product', 'variant', 'product_id', 'read_timestamp', 'file_name', 'file_size')

In [ ]:
df_silver.write\
    .format('delta') \
    .option('delta.enableChangeDataFeed', 'true') \
    .option('mergeSchema', 'true') \
    .mode('overwrite') \
    .saveAsTable(f'{catalog}.{silve_schema}.{data_source}')

In [ ]:
# ---> GOLD LAYER <---

df_silver = spark.sql(f'SELECT * FROM {catalog}.{silver_schema}.{data_source};')
df_gold = df_silver.select('product', 'product_id', 'division', 'category', 'product')

In [ ]:
df.gold.write\
    .format('delta') \
    .option('delta.enableChangeDataFeed', 'true') \
    .mode('overwrite') \
    .saveAsTable(f'{catalog}.{gold_schema}.sb_dim_{data_source}')

In [ ]:
# ---> MERGIN DATA SOURCE WITH PARENT <---
delta_table = DeltaTable.forName(spark, 'fmcg.gold_dim_products')
df_child_products = spark.sql(f'SELECT product_code, division, category, product, variant FROM fmcg.sb_dim_products')


In [ ]:
# ---> UPSERT APPROACH <---
delta_table.alias('parent').merge(
    source=df_child_products.alias('child'),
    condition='parent.product_code = child.product_code'
).whenMatchUpdate(
    set={
        'division': 'parent.division',
        'category': 'parent.category',
        'product': 'parent.category',
        'variant': 'parent.category'
    }
).whenNotMatchedInsert(
    values={
        'product_code': 'parent.product_code',
        'division': 'parent.division',
        'category': 'parent.category',
        'product': 'parent.product',
        'variant': 'parent.variant'
    }
).execute()